# Notebook 1 — the transfer equation and Beer–Lambert

Level-1 code: integrate $dI/ds = -\chi I$ by hand and compare with $I_0 e^{-\tau}$; then check `rtedu.slab.Slab`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))   # rtedu, uninstalled (education/src)
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI, GROUP_COLOUR
rng = np.random.default_rng(rtedu.SEEDS["ch01"])

## Level 1: integrate the pure-absorption transfer equation

The slab has geometric depth `depth` and a constant opacity `chi` per unit length. We step $I$ forward with explicit Euler, $I_{k+1} = I_k (1 - \chi\,\Delta s)$, and compare with the closed form.

In [ ]:
depth, chi = 2.0, 1.5              # cm and cm^-1: tau_total = 3
n_steps = 3000
ds = depth / n_steps
s = np.linspace(0.0, depth, n_steps + 1)
I = np.empty(n_steps + 1); I[0] = 1.0
for k in range(n_steps):
    I[k + 1] = I[k] * (1.0 - chi * ds)      # dI = -chi I ds
tau = chi * s
I_exact = np.exp(-tau)
euler_rel_error = float(abs(I[-1] - I_exact[-1]) / I_exact[-1])
print(f"tau_total = {tau[-1]:.2f}; Euler I/I0 = {I[-1]:.6f}; exact e^-tau = {I_exact[-1]:.6f}; rel. error {euler_rel_error:.2e}")

## Validation against `rtedu`

`Slab.transmission` is the closed form; the Euler curve must sit on it to the truncation error $\sim \chi\,\Delta s/2$ per unit optical depth.

In [ ]:
from rtedu.slab import Slab
slab = Slab(depth, chi)
assert np.isclose(slab.optical_depth(), tau[-1])
assert np.allclose(I, slab.transmission(s=s), rtol=5e-3)
mc = slab.mc_transmission(rng, 50_000)
sigma = np.sqrt(I_exact[-1] * (1 - I_exact[-1]) / 50_000)
print(f"Monte Carlo transmission {mc['transmitted']:.4f} vs {I_exact[-1]:.4f} (binomial sigma {sigma:.4f})")
assert abs(mc["transmitted"] - I_exact[-1]) < 4 * sigma

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(s, I, color=OI["blue"], label="Euler"); axes[0].plot(s, I_exact, "--", color=OI["black"], label=r"$e^{-\chi s}$")
axes[0].set_xlabel("distance s [cm]"); axes[0].set_ylabel(r"$I/I_0$"); axes[0].legend()
axes[1].plot(tau, I, color=OI["blue"]); axes[1].plot(tau, I_exact, "--", color=OI["black"]); axes[1].set_xlabel(r"optical depth $\tau = \chi s$"); axes[1].set_ylabel(r"$I/I_0$")
tt = np.linspace(0, 6, 200); axes[2].semilogy(tt, np.exp(-tt), color=OI["red"]); axes[2].axvline(tau[-1], color="grey", lw=0.8)
axes[2].plot([tau[-1]], [mc["transmitted"]], "o", color=OI["blue"], label="Monte Carlo, this slab"); axes[2].set_xlabel(r"$\tau$"); axes[2].set_ylabel(r"transmission $e^{-\tau}$"); axes[2].legend()
fig.suptitle("Beer-Lambert: the same curve against distance and against optical depth", fontsize=10); fig.tight_layout()
save_fig(fig, "ch01_beer_lambert")

In [ ]:
results.record("ch01", dict(depth_cm=depth, chi_per_cm=chi, tau_total=tau[-1], n_steps=n_steps,
                            transmission_exact=I_exact[-1], transmission_euler=I[-1], euler_rel_error=euler_rel_error,
                            transmission_mc=mc["transmitted"], mc_n=mc["n"], mc_sigma=sigma))